# Grouped Train, Validation and Test Split

In this notebook, I will divide the development dataset without placing
the same `chart_group_id` in more than one set.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_split import (
    create_grouped_split,
)

In [ ]:
dataset_path = (
    project_root
    / "data"
    / "processed"
    / "development_chart_claim_dataset.csv"
)

development_data = pd.read_csv(
    dataset_path
)

print("Rows:", len(development_data))
print(
    "Chart groups:",
    development_data[
        "chart_group_id"
    ].nunique(),
)

development_data.head()

## Split rule

All styles and all claims from one chart window must remain together.

The target row proportions are 70% train, 15% validation and 15% test.

In [ ]:
split_data, split_manifest = (
    create_grouped_split(
        data=development_data,
        group_column="chart_group_id",
        train_ratio=0.70,
        validation_ratio=0.15,
        test_ratio=0.15,
        random_state=42,
    )
)

for split_name, split_frame in (
    split_data.items()
):
    print(
        split_name,
        "rows:",
        len(split_frame),
        "groups:",
        split_frame[
            "chart_group_id"
        ].nunique(),
    )

In [ ]:
split_manifest

## Leakage check

The intersections below must be empty.

In [ ]:
train_groups = set(
    split_data["train"][
        "chart_group_id"
    ]
)

validation_groups = set(
    split_data["validation"][
        "chart_group_id"
    ]
)

test_groups = set(
    split_data["test"][
        "chart_group_id"
    ]
)

print(
    "Train and validation:",
    train_groups & validation_groups,
)
print(
    "Train and test:",
    train_groups & test_groups,
)
print(
    "Validation and test:",
    validation_groups & test_groups,
)

## Label balance

The three classes should remain balanced in every split.

In [ ]:
label_distribution = pd.DataFrame(
    {
        split_name: (
            split_frame["label"]
            .value_counts()
        )
        for split_name, split_frame
        in split_data.items()
    }
).reindex(
    [
        "supported",
        "refuted",
        "not_enough_information",
    ]
)

label_distribution

In [ ]:
split_sizes = pd.Series(
    {
        split_name: len(split_frame)
        for split_name, split_frame
        in split_data.items()
    }
)

plt.figure(figsize=(7, 4))
split_sizes.plot(kind="bar")

plt.title("Rows in Each Data Split")
plt.xlabel("Split")
plt.ylabel("Number of examples")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
processed_directory = (
    project_root
    / "data"
    / "processed"
)

file_names = {
    "train": (
        "train_chart_claim_dataset.csv"
    ),
    "validation": (
        "validation_chart_claim_dataset.csv"
    ),
    "test": (
        "test_chart_claim_dataset.csv"
    ),
}

for split_name, file_name in (
    file_names.items()
):
    split_data[split_name].to_csv(
        processed_directory / file_name,
        index=False,
    )

split_manifest.to_csv(
    processed_directory
    / "data_split_manifest.csv",
    index=False,
)

print("Split files saved.")

## Result

The split contains 252 training rows, 54 validation rows and 54 test
rows.

No chart group appears in more than one set. The next notebook can use
the training set for baseline models while the test set remains
untouched.